# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## ML Project Iteration

After getting the resulting models from the Genetic Algorithm, using them in the completed versions (first versions) of the frontend and the backend, and analyzing the results in the Draft Report, this is the next step in trying to improve the models used based on the results. While I previously thought that the GA would potentially provide the best results, the KerasTuner, which was only intended to be used to obtain the structure of the model to be used, proved to produce the best MAE.

The result of this notebook are 3 models per currency pair (each with a delay longer than the previous one), using the KerasTuner procedure, used in the ml_process notebook.

## Preparing the data

### Transforming the csv data to a numpy array

In [1]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

usdYen_raw_data = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
usdYen_raw_data_dates = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})

# As the currency data is from newer to older, the order should be inverted.
usdYen_raw_data = np.flip(usdYen_raw_data, axis=0)

# Computing the numer of samples for each data split
train_samples_number = len(usdYen_raw_data)

# To check that eveything works as expected
# print("Length: ",len(usdYen_raw_data))
# print("Data type: ",usdYen_raw_data.dtype)
# print("Raw Data: ",usdYen_raw_data)
# print("Raw Data Dates: ",usdYen_raw_data_dates)
# print("Number of train samples: ", train_samples_number)

### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [2]:
from tensorflow import keras

# Parameters
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delay = sampling_rate * (sequence_length + 5 - 1) # target is 5 days after the end of the sequence
batch_size = train_samples_number

# train dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    usdYen_raw_data,
    targets=usdYen_raw_data[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

# Check that the timeseries data works correctly
# for inputs, targets in train_dataset:
#    for i in range(inputs.shape[0]):
#        print([float(x) for x in inputs[i]], float(targets[i]))

### - Extracting data inputs and outputs

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [3]:
import tensorflow as tf

data_inputs = []
data_outputs = []

for samples, targets in train_dataset:
    # print("Samples: ", samples)
    # print("Sample shape: ", samples.shape)
    # print("Targets: ", targets)
    # print("Targets shape: ", targets.shape)
    data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
    data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

data_inputs_test = data_inputs[-200:]
data_outputs_test = data_outputs[-200:]
data_inputs = data_inputs[:-200]
data_outputs = data_outputs[:-200]

# Check that the data inputs and outputs work correctly
# print("Data Inputs: ", len(data_inputs))
# print("Data Inputs Test: ", len(data_inputs_test))
# print("Data Outputs: ", len(data_outputs))
# print("Data Outputs Test: ", len(data_outputs_test))
    

2026-02-14 17:55:20.853815: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### - Test Model resulting from GA (USD/JPY - 5 days delay)

In [4]:
lstm_model_loaded = keras.models.load_model("models/model_jpy_delay_5.keras")
print(f"Test MAE: {lstm_model_loaded.evaluate(data_inputs_test, data_outputs_test)[1]}")

/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 11 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 371.2234 - mae: 18.7104  
Test MAE: 18.710359573364258


## Get structure of the best model

Get the structure of the best model obtained by using KerasTuner in the ml_process notebook.

In [5]:
from keras.models import load_model

loaded_model = load_model("models/ml_process_tuner_best_model.keras")

loaded_model.summary()

# Get which activation functions were used
for layer in loaded_model.layers:
    print(layer.activation)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 28)             │           924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            29 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,860 (155.71 KB)

 Trainable params: 19,929 (77.85 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,931 (77.86 KB)

<function tanh at 0x12d2af100>
<function relu at 0x12bdbbce0>
<function relu at 0x12bdbbce0>
<function linear at 0x12d2af7e0>


Create a new model based on the structure of the best model

In [6]:
from keras import models
from keras import layers
from keras import activations

def build_lstm_model(hp):
    model = models.Sequential()
    model.add(layers.LSTM(64, input_shape=(sequence_length, 1)))
    model.add(layers.Dense(32, activation=activations.relu))
    model.add(layers.Dense(28, activation=activations.relu))
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

## Preparing the data for KerasTuner

### Currencies

In [7]:
currencies = [
    "jpy",
    "aud",
    "cad",
    "cny",
    "hkd",
    "chf",
]

### Transforming the csv data to a numpy array

In [8]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

currencies_raw_data = {}

# Pairs of currencies
for currency in currencies:
    raw_data = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
    raw_data_dates = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})
    
    # As the currency data is from newer to older, the order should be inverted.
    raw_data = np.flip(raw_data, axis=0)
    raw_data_dates = np.flip(raw_data_dates, axis=0)

    # calculate train sample number
    train_samples_number = len(raw_data)

    currencies_raw_data[f"{currency}"] = {
        f"usd_{currency}_train_samples_number": train_samples_number,
        f"usd_{currency}_raw_data": raw_data,
        f"usd_{currency}_raw_data_dates": raw_data_dates
    }

    # Print essential data information
    # print(f"Length usd_{currency}_raw_data: ",len(raw_data))
    # print(f"Data type usd_{currency}_raw_data: ",raw_data.dtype)
    # print(f"Raw Data Sample - usd_{currency}_raw_data: ",raw_data[:10])
    # print(f"Raw Data Dates Sample - usd_{currency}_raw_data_dates: ",raw_data_dates[:10])
    # print(f"Number of train samples - usd_{currency}_raw_data_dates: ", train_samples_number)

# Check one of the currencies in the dictionary
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data"][:10])
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data_dates"][:10])

# print(currencies_raw_data)

### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [9]:
from tensorflow import keras
import tensorflow as tf

# Parameters: sampling_rate, sequence_length, delay, and batch_size
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delays = []

for i in range(3):
    delays.append(sampling_rate * (sequence_length + (5 + (20 * i)) - 1)) # target is 5, 25, 45 days after the end of the sequence

train_datasets = {}

data_inputs_test_all = {}
data_outputs_test_all = {}
data_inputs_all = {}
data_outputs_all = {}

for index, delay in enumerate(delays):
    for currency in currencies:
        # train dataset
        train_dataset = keras.utils.timeseries_dataset_from_array(
            currencies_raw_data[currency][f"usd_{currency}_raw_data"],
            targets=currencies_raw_data[currency][f"usd_{currency}_raw_data"][delay:],
            sampling_rate=sampling_rate,
            sequence_length=sequence_length,
            batch_size=currencies_raw_data[currency][f"usd_{currency}_train_samples_number"],
        )

        if index == 0:
            train_datasets[f"train_dataset_usd_{currency}_delay_5"] = train_dataset
        if index == 1:
            train_datasets[f"train_dataset_usd_{currency}_delay_25"] = train_dataset
        if index == 2:
            train_datasets[f"train_dataset_usd_{currency}_delay_45"] = train_dataset

        # Extracting data inputs and outputs
        # Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].
        
        for samples, targets in train_dataset:
            # print("Samples: ", samples)
            # print("Sample shape: ", samples.shape)
            # print("Targets: ", targets)
            # print("Targets shape: ", targets.shape)
            data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
            data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

            if index == 0:
                data_inputs_test_all[f"{currency}_delay_5"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_5"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_5"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_5"] = data_outputs[:-200]
            if index == 1:
                data_inputs_test_all[f"{currency}_delay_25"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_25"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_25"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_25"] = data_outputs[:-200]
            if index == 2:
                data_inputs_test_all[f"{currency}_delay_45"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_45"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_45"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_45"] = data_outputs[:-200]

2026-02-14 17:55:21.666177: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-14 17:55:22.145093: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-14 17:55:23.083917: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-14 17:55:24.898271: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### - Kerastuner implementation

The code is based on the code provided in the Keras documentation for the KerasTuner [2].

In [10]:
import keras_tuner
import os

# using KerasTuner to get three models per currency
for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")

        tuner = keras_tuner.RandomSearch(
            hypermodel=build_lstm_model,
            objective="val_mae",
            max_trials=1,
            executions_per_trial=5,
            overwrite=True,
            directory="tuner_results/final_project_ml_project_iteration_results",
            project_name="final_project_tuner_ml_project_iteration"
        )

        # check if ml_process_tuner_best_model exists
        is_ml_process_tuner_best_model_created = os.path.exists(f"models/ml_project_iteration_best_model_{currency}_delay_{real_delay}.keras")

        # if it does not exists run the tuner
        if is_ml_process_tuner_best_model_created != True:
            tuner.search(data_inputs_all[f"{currency}_delay_{real_delay}"], data_outputs_all[f"{currency}_delay_{real_delay}"], epochs=20, validation_data=(data_inputs_test_all[f"{currency}_delay_{real_delay}"], data_outputs_test_all[f"{currency}_delay_{real_delay}"]))
            best_model = tuner.get_best_models(num_models=1)[0]
            best_model.summary()

            # save model
            best_model.save(f"models/ml_project_iteration_best_model_{currency}_delay_{real_delay}.keras")

Trial 1 Complete [00h 07m 00s]
val_mae: 0.014657742157578469

Best val_mae So Far: 0.014657742157578469
Total elapsed time: 00h 07m 00s


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 28)             │           924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            29 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,929 (77.85 KB)

 Trainable params: 19,929 (77.85 KB)

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 28)             │           924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            29 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,929 (77.85 KB)

 Trainable params: 19,929 (77.85 KB)

 Non-trainable params: 0 (0.00 B)

## Preparing the data for GA and Kerastuner models evaluation

### Currencies

In [1]:
currencies = [
    "jpy",
    "aud",
    "cad",
    "cny",
    "hkd",
    "chf",
]

### Transforming the csv data to a numpy array

In [2]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

currencies_raw_data = {}

# Pairs of currencies
for currency in currencies:
    raw_data = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
    raw_data_dates = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})
    
    # As the currency data is from newer to older, the order should be inverted.
    raw_data = np.flip(raw_data, axis=0)
    raw_data_dates = np.flip(raw_data_dates, axis=0)

    # calculate train sample number
    train_samples_number = len(raw_data)

    currencies_raw_data[f"{currency}"] = {
        f"usd_{currency}_train_samples_number": train_samples_number,
        f"usd_{currency}_raw_data": raw_data,
        f"usd_{currency}_raw_data_dates": raw_data_dates
    }

    # Print essential data information
    # print(f"Length usd_{currency}_raw_data: ",len(raw_data))
    # print(f"Data type usd_{currency}_raw_data: ",raw_data.dtype)
    # print(f"Raw Data Sample - usd_{currency}_raw_data: ",raw_data[:10])
    # print(f"Raw Data Dates Sample - usd_{currency}_raw_data_dates: ",raw_data_dates[:10])
    # print(f"Number of train samples - usd_{currency}_raw_data_dates: ", train_samples_number)

# Check one of the currencies in the dictionary
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data"][:10])
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data_dates"][:10])

# print(currencies_raw_data)

### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [3]:
from tensorflow import keras
import tensorflow as tf

# Parameters: sampling_rate, sequence_length, delay, and batch_size
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delays = []

for i in range(3):
    delays.append(sampling_rate * (sequence_length + (5 + (20 * i)) - 1)) # target is 5, 25, 45 days after the end of the sequence

train_datasets = {}

data_inputs_test_all = {}
data_outputs_test_all = {}
data_inputs_all = {}
data_outputs_all = {}

for index, delay in enumerate(delays):
    for currency in currencies:
        # train dataset
        train_dataset = keras.utils.timeseries_dataset_from_array(
            currencies_raw_data[currency][f"usd_{currency}_raw_data"],
            targets=currencies_raw_data[currency][f"usd_{currency}_raw_data"][delay:],
            sampling_rate=sampling_rate,
            sequence_length=sequence_length,
            batch_size=currencies_raw_data[currency][f"usd_{currency}_train_samples_number"],
        )

        if index == 0:
            train_datasets[f"train_dataset_usd_{currency}_delay_5"] = train_dataset
        if index == 1:
            train_datasets[f"train_dataset_usd_{currency}_delay_25"] = train_dataset
        if index == 2:
            train_datasets[f"train_dataset_usd_{currency}_delay_45"] = train_dataset

        # Extracting data inputs and outputs
        # Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].
        
        for samples, targets in train_dataset:
            # print("Samples: ", samples)
            # print("Sample shape: ", samples.shape)
            # print("Targets: ", targets)
            # print("Targets shape: ", targets.shape)
            data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
            data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

            if index == 0:
                data_inputs_test_all[f"{currency}_delay_5"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_5"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_5"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_5"] = data_outputs[:-200]
            if index == 1:
                data_inputs_test_all[f"{currency}_delay_25"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_25"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_25"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_25"] = data_outputs[:-200]
            if index == 2:
                data_inputs_test_all[f"{currency}_delay_45"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_45"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_45"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_45"] = data_outputs[:-200]

2026-02-22 12:13:53.389680: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-22 12:13:53.575639: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-22 12:13:53.941844: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-22 12:13:54.665731: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-02-22 12:13:56.128871: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### - Kerastuner resulting models evaluation

In [10]:
from tabulate import tabulate

# Evaluation and comparison between the MAE of the GA models and the MAE of ML Project Iteration (KerasTuner) models
delay_five_table = []
delay_twentyfive_table = []
delay_fortyfive_table = []

for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    # print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")
        ga_model_loaded = keras.models.load_model(f"models/model_{currency}_delay_{real_delay}.keras")
        lstm_custom_model_loaded = keras.models.load_model(f"models/ml_project_iteration_best_model_{currency}_delay_{real_delay}.keras")
        test_mae_ga_model_loaded = ga_model_loaded.evaluate(data_inputs_test_all[f"{currency}_delay_{real_delay}"], data_outputs_test_all[f"{currency}_delay_{real_delay}"])[1]
        test_mae_lstm_custom_model_loaded = lstm_custom_model_loaded.evaluate(data_inputs_test_all[f"{currency}_delay_{real_delay}"], data_outputs_test_all[f"{currency}_delay_{real_delay}"])[1]
        if real_delay == 5:
            delay_five_table.append([f"USD/{currency.upper()}", test_mae_ga_model_loaded, test_mae_lstm_custom_model_loaded])
        if real_delay == 25:
            delay_twentyfive_table.append([f"USD/{currency.upper()}",test_mae_ga_model_loaded, test_mae_lstm_custom_model_loaded])
        if real_delay == 45:
            delay_fortyfive_table.append([f"USD/{currency.upper()}",test_mae_ga_model_loaded, test_mae_lstm_custom_model_loaded])


---- ---- ---- ----
Currency: jpy.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 371.2234 - mae: 18.7104 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 5.3318 - mae: 1.9388 
---- ---- ---- ----
Currency: aud.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2192 - mae: 0.4663 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.7801e-04 - mae: 0.0106 
---- ---- ---- ----
Currency: cad.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0023 - mae: 0.0446  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 8.4491e-05 - mae: 0.0075  
---- ---- ---- ----
Currency: cny.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0021 - mae: 0.0392  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4.0277e-04 - mae: 0.0155 
---- ---- ---- ----
Currency: hkd.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0207 - mae: 0.1408 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0011 - mae: 0.0288 
---- ---- ---- ----
Currency: chf.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0256 - mae: 0.1599 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s

Create the tables to compare the results

In [11]:
# Create the table for the 5 day delay
print(tabulate(delay_five_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration (KerasTuner)"], tablefmt="grid"))

+------------+------------+-------------------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration (KerasTuner) |
+============+============+===========================================+
| USD/JPY    | 18.7104    |                                1.93882    |
+------------+------------+-------------------------------------------+
| USD/AUD    |  0.466277  |                                0.0105518  |
+------------+------------+-------------------------------------------+
| USD/CAD    |  0.0446317 |                                0.00746274 |
+------------+------------+-------------------------------------------+
| USD/CNY    |  0.0391992 |                                0.0155393  |
+------------+------------+-------------------------------------------+
| USD/HKD    |  0.140809  |                                0.0288446  |
+------------+------------+-------------------------------------------+
| USD/CHF    |  0.159853  |                                0.006

In [12]:
# Create the table for the 25 day delay
print(tabulate(delay_twentyfive_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration (KerasTuner)"], tablefmt="grid"))

+------------+------------+-------------------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration (KerasTuner) |
+============+============+===========================================+
| USD/JPY    | 28.249     |                                 2.83452   |
+------------+------------+-------------------------------------------+
| USD/AUD    |  1.13943   |                                 0.0155922 |
+------------+------------+-------------------------------------------+
| USD/CAD    |  0.0404597 |                                 0.0157466 |
+------------+------------+-------------------------------------------+
| USD/CNY    |  0.605407  |                                 0.037116  |
+------------+------------+-------------------------------------------+
| USD/HKD    |  0.0701988 |                                 0.0377332 |
+------------+------------+-------------------------------------------+
| USD/CHF    |  0.0684906 |                                 0.01

In [13]:
# Create the table for the 45 day delay
print(tabulate(delay_fortyfive_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration (KerasTuner)"], tablefmt="grid"))

+------------+------------+-------------------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration (KerasTuner) |
+============+============+===========================================+
| USD/JPY    | 23.3651    |                                 3.52465   |
+------------+------------+-------------------------------------------+
| USD/AUD    |  0.0968492 |                                 0.0156326 |
+------------+------------+-------------------------------------------+
| USD/CAD    |  0.232504  |                                 0.0191894 |
+------------+------------+-------------------------------------------+
| USD/CNY    |  0.289527  |                                 0.0322847 |
+------------+------------+-------------------------------------------+
| USD/HKD    |  0.0312488 |                                 0.0339194 |
+------------+------------+-------------------------------------------+
| USD/CHF    |  0.198467  |                                 0.01

## About the code

The timeseries code is based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

The KerasTuner code is based on the code provided in the Keras documentation for the KerasTuner [2].

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Preparing the data. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_5

2- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/